# 03. CIFAR-10 컬러 이미지 분류

CIFAR-10은 비행기, 자동차, 동물 등 **10개 클래스**로 구성된 컬러 이미지 데이터셋이다. 이미지 한 장의 크기는 작지만 배경과 물체의 모양이 다양하므로, FashionMNIST보다 분류 난도가 높은 편이다.

- CIFAR: Canadian Institute For Advanced Research(캐나다 고등연구소)
- 데이터 수: 훈련 이미지 50,000장, 테스트 이미지 10,000장
- 이미지 형태: RGB 3채널, 32×32 픽셀
- 클래스 종류

```python
class_names = [
    "airplane",   # 비행기
    "automobile", # 자동차
    "bird",       # 새
    "cat",        # 고양이
    "deer",       # 사슴
    "dog",        # 개
    "frog",       # 개구리
    "horse",      # 말
    "ship",       # 배
    "truck",      # 트럭
]
```

## FashionMNIST에서 달라지는 점

- FashionMNIST는 `(N, 1, 28, 28)`의 흑백 의류 이미지였지만, CIFAR-10은 `(N, 3, 32, 32)`의 RGB 사물 이미지이다.
- 첫 번째 `Conv2d`의 `in_channels`가 1에서 3으로 바뀐다. 출력 필터 하나는 R·G·B용 커널 3개를 함께 사용해 하나의 feature map을 만든다.
- 색상, 배경과 물체 형태가 다양하므로 더 많은 합성곱층과 데이터 증강을 사용할 수 있다.
- 데이터 종류가 달라도 `데이터 준비 → 모델 정의 → 손실 계산 → 역전파 → 평가`라는 PyTorch 학습 흐름은 같다.

## 핵심 개념

- **입력 Tensor**: CIFAR-10 한 장은 `(3, 32, 32)`, 미니배치는 `(N, 3, 32, 32)` 형태이다.
- **Normalize**: 각 RGB 채널의 값을 비슷한 범위로 조정하여 학습을 안정화한다.
- **CNN**: 합성곱층은 지역 특징을 찾고, ReLU는 비선형성을 추가하며, pooling은 공간 크기를 줄인다.
- **logits**: 마지막 출력층이 만드는 클래스별 원점수이다. `CrossEntropyLoss`에는 softmax를 적용하지 않은 logits를 전달한다.
- **학습 흐름**: 미니배치마다 `zero_grad → forward → loss → backward → step`을 반복한다.

## 1. CIFAR-10 데이터셋과 전처리

`CIFAR10` Dataset은 이미지와 정답을 한 쌍으로 관리하며, `trainset[index]`처럼 샘플을 꺼낼 때 지정한 `transform`을 적용한다.

- `trainset.data`: 전처리 전 원본 NumPy 배열이며 축 순서는 `(N, H, W, C)`이다.
- `trainset[index]`: 전처리가 적용된 `(이미지 Tensor, 클래스 번호)`이며 이미지 축 순서는 `(C, H, W)`이다.
- `transforms.Compose()`: 여러 전처리를 작성된 순서대로 연결한다.
- `transforms.ToTensor()`: 축을 `(H, W, C)`에서 `(C, H, W)`로 바꾸고, 자료형을 `float32`, 픽셀 범위를 `0~255`에서 `0~1`로 변환한다.
- `transforms.Normalize(mean, std)`: RGB 채널마다 `(픽셀값 - 평균) / 표준편차`를 계산한다. 여기서는 평균과 표준편차를 모두 `0.5`로 두어 `0~1` 값을 약 `-1~1`로 바꾼다.

`0.5`는 CIFAR-10 전체에서 직접 계산한 실제 채널 평균·표준편차가 아니라, 수업에서 값의 중심을 0 근처로 옮기기 위한 간단한 기준이다. `Normalize()`는 Tensor를 입력으로 사용하므로 반드시 `ToTensor()` 뒤에 작성한다.

`download=True`이므로 데이터가 없으면 `./data`에 자동으로 내려받으며, 이미 있으면 저장된 파일을 다시 사용한다.

In [ ]:
import torch
from torchvision.datasets import CIFAR10
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5),
    ),
])

trainset = CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

testset = CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform,
)

print("훈련 원본 배열 shape:", trainset.data.shape)
print("테스트 원본 배열 shape:", testset.data.shape)
print("클래스:", trainset.classes)


### Dataset을 미니배치로 묶는 DataLoader

`Dataset`이 이미지 한 장과 정답 하나를 관리한다면, `DataLoader`는 여러 샘플을 미니배치로 묶어 모델에 전달한다.

- `batch_size=64`: 한 번의 순전파와 가중치 갱신에 최대 64장을 사용한다.
- `shuffle=True`: epoch마다 훈련 데이터 순서를 섞어 특정 데이터 순서를 외우는 현상을 줄인다.
- 이미지 배치 `(N, 3, 32, 32)`의 `N`은 현재 미니배치의 이미지 수이다.
- 정답 배치 `(N,)`에는 각 이미지에 대응하는 0~9의 클래스 index가 들어 있다.

이 셀의 `trainloader`는 전체 훈련 데이터를 살펴보기 위한 임시 DataLoader이다. 뒤에서 train·validation을 분리한 후 실제 학습용 `train_loader`를 다시 만든다.

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64
trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)

testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False)

images, labels = next(iter(trainloader))

print("이미지 배치 shape:", images.shape)
print("정답 배치 shape:", labels.shape)
print("첫 번째 이미지 shape:", images[0].shape)
print("첫 번째 정답 index:", labels[0].item())
print("첫 번째 정답 이름:", trainset.classes[labels[0].item()])
print("정규화된 값 범위:", images.min().item(), "~", images.max().item())


## 2. 컬러 이미지와 클래스 시각화

모델에는 정규화된 `(C, H, W)` Tensor를 넣지만, Matplotlib은 `0~1` 범위의 `(H, W, C)` 배열을 표시하기 편하다. 따라서 시각화할 때만 모델 전처리를 반대로 되돌린다.

- **역정규화**: `정규화값 × 0.5 + 0.5`로 약 `-1~1` 값을 다시 `0~1` 범위로 복원한다.
- **축 변환**: `permute(1, 2, 0)`으로 `(C, H, W)`를 `(H, W, C)`로 바꾼다.
- 이 변환은 화면에 정상적인 색으로 표시하기 위한 것이며, 모델 입력에는 정규화된 Tensor를 그대로 사용한다.

In [ ]:
import matplotlib.pyplot as plt

sample_images, sample_labels = next(iter(trainloader))

fig, axes = plt.subplots(2, 8, figsize=(18, 6))

for index, axis in enumerate(axes.flat):

    image = sample_images[index].permute(1, 2, 0)

    image = image * 0.5 + 0.5

    image = torch.clamp(image, 0, 1)

    axis.imshow(image.numpy())
    axis.set_title(trainset.classes[sample_labels[index].item()])
    axis.axis("off")

plt.tight_layout()
plt.show()


### 데이터 증강

**데이터 증강(Data Augmentation)** 은 이미지의 정답은 유지하면서 위치, 방향, 크기 등을 조금씩 바꾸어 학습 데이터의 다양성을 높이는 방법이다.
새로운 이미지 파일을 저장하는 것이 아니라, Dataset에서 이미지를 꺼낼 때마다 무작위 변환을 적용한다.

- `RandomCrop`: 이미지 일부를 무작위 위치에서 잘라 물체의 위치가 조금씩 달라지게 한다.
- `RandomHorizontalFlip`: 이미지를 일정 확률로 좌우 반전한다.

이렇게 하면 모델이 특정 위치나 방향을 외우는 과적합을 줄이고, 새로운 이미지에 대한 성능을 높일 수 있다. 일반적으로 학습 데이터에만 적용하고 validation·test 데이터에는 적용하지 않는다. 원본 흐름을 유지하기 위해 아래에서는 효과만 확인하고 기본 학습 데이터에는 적용하지 않는다.

In [ ]:
raw_image = trainset.data[0]

# 데이터 증강


# ----------------


fig, axes = plt.subplots(1, 5, figsize=(12, 3))

axes[0].imshow(raw_image)
axes[0].set_title("original")
axes[0].axis("off")

for index in range(1, 5):
    augmented_image = augmentation(raw_image)
    axes[index].imshow(augmented_image)
    axes[index].set_title(f"augmented {index}")
    axes[index].axis("off")

plt.tight_layout()
plt.show()


## 3. CIFAR-10 CNN 모델

FashionMNIST보다 색상과 배경이 다양한 CIFAR-10을 처리하기 위해 합성곱 층(Conv2d * 2) 블록을 세 번 사용한다.
각 블록은 `Conv2d → ReLU`를 두 번 적용해 RGB 픽셀에서 색상 변화, 경계선, 무늬, 질감 등의 특징을 충분히 추출한다.
그 뒤 `MaxPool2d`로 높이와 너비를 절반으로 줄인다.

- 첫 번째 블록: `(N, 3, 32, 32) → (N, 32, 16, 16)`
- 두 번째 블록: `(N, 32, 16, 16) → (N, 64, 8, 8)`
- 세 번째 블록: `(N, 64, 8, 8) → (N, 128, 4, 4)`
- Flatten: `(N, 128, 4, 4) → (N, 2048)`
- 완전연결층: 추출된 2,048개 특징을 종합해 클래스별 logits 10개를 만든다.

공간 크기는 `32 → 16 → 8 → 4`로 줄이고 채널 수는 `3 → 32 → 64 → 128`로 늘린다. 위치 정보는 압축하면서 물체를 구분할 수 있는 특징의 종류를 늘리는 흐름이다.

### logits shape와 확률 확인

모델은 이미지 한 장마다 10개의 logits를 출력한다. logit은 확률이 아니라 클래스별 원점수이므로 음수일 수 있고 합이 1일 필요도 없다.

- 학습할 때: `CrossEntropyLoss(logits, labels)`에 logits를 그대로 전달한다.
- 예측할 때: `argmax(dim=1)`로 가장 큰 logit의 클래스 index를 선택한다.
- 확률이 필요할 때: `softmax(logits, dim=1)`로 각 이미지의 10개 점수를 합이 1인 값으로 변환한다.

아직 학습 전이므로 아래 예측에는 의미가 없다. 이 셀의 목적은 모델의 입력·출력 shape와 계산 가능 여부를 학습 전에 점검하는 것이다.

In [ ]:
dummy_images = torch.randn(4, 3, 32, 32)

model.eval()
with torch.no_grad():
    logits = model(dummy_images)

    probabilities = torch.softmax(logits, dim=1)

print("logits shape:", logits.shape)
print("첫 이미지 logits:", logits[0])
print("첫 이미지 확률 합:", probabilities[0].sum().item())
print("예측 클래스 index:", logits[0].argmax().item())

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)
print("학습 가능한 파라미터 수:", trainable_parameters)


## 4. 학습·검증 데이터와 학습 도구 준비

공식 훈련 데이터 50,000장을 실제 학습용 train 85%와 학습 상태 점검용 validation 15%로 나눈다. 공식 test 데이터는 모델 선택에 사용하지 않고 마지막 최종 평가 때만 사용한다.

- **train**: `backward()`와 `optimizer.step()`을 실행해 가중치를 변경한다.
- **validation**: 가중치를 변경하지 않고 epoch마다 과적합과 일반화 상태를 확인한다.
- **test**: 학습과 모델 선택이 끝난 뒤 최종 성능을 한 번 평가한다.
- **device**: 모델과 입력 Tensor가 계산될 CPU·GPU·MPS 장치이다. 모델과 데이터는 같은 device에 있어야 한다.
- **CrossEntropyLoss**: `(N, 10)` logits와 `(N,)` 정수 클래스 index를 비교한다.
- **Adam**: `backward()`가 계산한 기울기를 이용해 모델의 가중치와 편향을 갱신한다.

앞에서 만든 `trainloader`는 데이터 확인용이고, 아래의 `train_loader`와 `val_loader`가 실제 학습·검증에 사용된다.

In [ ]:
from torch.utils.data import random_split

train_size = int(0.85 * len(trainset))
val_size = len(trainset) - train_size

split_generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(
    trainset,
    [train_size, val_size],
    generator=split_generator,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

print("device:", device)
print("학습 데이터 수:", len(train_dataset))
print("검증 데이터 수:", len(val_dataset))
print("테스트 데이터 수:", len(testset))


### CNN 학습 루프

하나의 **epoch**는 train 데이터 전체를 한 번 학습하는 단위이다. 각 epoch 안에서 미니배치마다 다음 순서를 반복한다.

1. `model.train()`: Dropout을 사용하는 학습 모드로 전환한다.
2. `optimizer.zero_grad()`: 이전 미니배치의 기울기를 지운다.
3. `outputs = model(images)`: 현재 가중치로 logits를 계산한다.
4. `loss = criterion(outputs, labels)`: 예측과 정답의 차이를 하나의 손실값으로 계산한다.
5. `loss.backward()`: 손실을 각 파라미터로 미분해 기울기를 계산한다.
6. `optimizer.step()`: 기울기를 이용해 가중치와 편향을 갱신한다.

train 학습이 끝나면 `model.eval()`과 `torch.no_grad()`로 validation을 평가한다. validation에는 `backward()`와 `step()`이 없으므로 가중치가 바뀌지 않는다. `EPOCHS`만 바꾸면 짧은 실행 확인과 실제 학습 시간을 조절할 수 있다.

## 5. 학습 곡선 확인

훈련·검증 손실과 정확도를 함께 그리면 학습 진행과 과적합 여부를 숫자 목록보다 쉽게 판단할 수 있다.

- train loss와 validation loss가 함께 감소하면 새로운 데이터에서도 학습이 진행되는 상태이다.
- train accuracy만 계속 상승하고 validation accuracy가 정체되면 학습 데이터를 외우는 과적합을 의심한다.
- train loss는 감소하지만 validation loss가 다시 상승하면 그 이전 epoch의 모델이 더 적절할 수 있다.
- loss가 크게 진동하거나 `NaN`이 되면 입력 값 범위, 학습률, label dtype과 logits shape를 확인한다.

정확도만 보지 않고 손실과 train·validation 사이의 간격을 함께 해석하는 것이 중요하다.

In [ ]:
epoch_numbers = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epoch_numbers, history["train_loss"], marker="o", label="train loss")
plt.plot(epoch_numbers, history["val_loss"], marker="o", label="validation loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Loss")
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epoch_numbers, history["train_acc"], marker="o", label="train accuracy")
plt.plot(epoch_numbers, history["val_acc"], marker="o", label="validation accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Accuracy")
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()


## 6. 테스트 데이터 최종 평가

테스트 데이터는 학습과 모델 선택에 사용하지 않은 데이터이다. 학습이 끝난 모델을 `eval()` 모드로 바꾸고, `torch.no_grad()` 안에서 평균 손실과 정확도를 계산한다.

- validation 성능: 학습 도중 epoch 수나 모델 설정을 판단하는 기준이다.
- test 성능: 모든 선택이 끝난 뒤 처음 보는 데이터에 대한 최종 일반화 성능이다.
- test 결과를 보고 반복해서 모델을 수정하면 test 데이터도 사실상 모델 선택에 사용한 것이 되므로 주의한다.

In [ ]:
model.eval()
test_loss_sum = 0.0
test_correct = 0

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss_sum += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        test_correct += (predictions == labels).sum().item()

test_loss = test_loss_sum / len(testloader.dataset)
test_acc = test_correct / len(testloader.dataset)

print(f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")


## 핵심 정리

1. CIFAR-10 이미지는 `(N, 3, 32, 32)` 형태의 RGB Tensor로 모델에 입력된다.
2. `Normalize`는 RGB 채널의 값 범위를 조정해 학습을 안정화한다.
3. 합성곱층은 특징을 찾고, ReLU는 비선형성을 추가하며, pooling은 공간 크기를 줄인다.
4. 마지막 선형층은 10개 클래스의 **logits**를 출력한다.
5. `CrossEntropyLoss`에는 softmax를 적용하지 않은 logits와 정수 정답 index를 전달한다.
6. 학습은 `zero_grad → forward → loss → backward → step` 순서로 진행한다.
7. validation은 학습 상태를 점검하고, test는 최종 일반화 성능을 평가한다.